# M4 Fix 2 — Independent 16-matrix GQDP audit

**No access to iService `anon_user` or its repository is required. No model retraining.** Uses your verified 395 MB M4 archive (or the existing imported project on Runpod), three saved O/U seeds, and the pinned `Qwen2.5-0.5B-Instruct` base model revision.

**Question:** Does the endpoint **GQDP** diagnostic from the original manuscript predict the measured **isolated, single-matrix** quantization-induced change of simulated obsolete-answer CE in M4?

**Design fixed before executing this extension:** 16 named matrices = depths 2, 8, 15, 22 × projections q, o, up, down. Three seeds × O/U × packed NF4/INT8 = **192 probes**. On a native BF16 NVIDIA A100, each chosen projection is tested twice: (a) quantized/dequantized dense-weight substitution matching the historical GT style and (b) an actual bitsandbytes packed module; other projections remain BF16. Gradient is computed at the original merged checkpoint on 40 **development** obsolete-answer examples; quantization response is scored on those same 40 development examples. The sealed M4 final test is not read or tuned against.

**Definitions preserve manuscript §3.2 / Supplement S1:** `GQDP_oriented = -(∇W L_tok) · (W_quantized_then_dequantized − W)`; `Y_dense = −(L_ex(single quantized/dequantized dense projection) − L_ex(dense model))`; a separate `Y_packed` measures actual one-module packed execution. `L_tok` weights all answer tokens equally; `L_ex` weights examples equally. These estimands are intentionally distinct, as in the historical study.

**Limit:** This is a post-main-study, exploratory local diagnostic audit, not a new unlearning algorithm, not confirmation that NPO succeeded, not a proof of diagnostic portability. Six seed-role models are the principal experimental units; 16 matrix measurements within one model are dependent. The earlier full-model packed execution experiment remains separate; the dense-dequant comparison tests the manuscript-style matrix response, while the packed comparison isolates execution differences.

## 1. Install the same model stack (once per Pod)

Runpod PyTorch/Jupyter template recommended. Estimated GPU cost at the previously advertised **2×A100 ~$3.58/hour**, plus storage; check the live Pod rate and remaining balance before starting. No duration is guaranteed. Avoid Spot Pods; don't deploy a GPU until ready to upload the archive.

In [ ]:
import sys,subprocess
subprocess.run([sys.executable, '-m','pip','install','-q',
    'transformers==4.51.3','peft==0.15.2','accelerate==1.6.0',
    'safetensors>=0.5.0','sentencepiece>=0.2.0','scipy>=1.11',
    'numpy>=1.24'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir',
    '--force-reinstall','--no-deps','bitsandbytes==0.50.2'],check=True)
print('Dependencies installed; using existing CUDA PyTorch runtime')

## 2. Restore the saved M4 project (no training)

If `/workspace/M4_IMPORTED_V2/` still exists, it is reused. Otherwise upload `M4_V2_COMPLETE_RESULTS.tar.gz` (approximately 395 MB) to the JupyterLab `/workspace` directory; the cell extracts it to `/workspace/M4_FIX2_INPUT/M4_v2`. It never changes the original archive or frozen result files.

In [ ]:
from pathlib import Path
import tarfile
OLD = Path('/workspace/M4_IMPORTED_V2')
NEW = Path('/workspace/M4_FIX2_INPUT/M4_v2')
ARCHIVE = Path('/workspace/M4_V2_COMPLETE_RESULTS.tar.gz')
if (OLD/'FINAL_TEST_FREEZE.json').exists():
    ROOT = OLD
elif (NEW/'FINAL_TEST_FREEZE.json').exists():
    ROOT = NEW
else:
    assert ARCHIVE.is_file(), 'Upload M4_V2_COMPLETE_RESULTS.tar.gz to /workspace/ via JupyterLab'
    stage=Path('/workspace/M4_FIX2_INPUT');stage.mkdir(parents=True,exist_ok=True)
    with tarfile.open(ARCHIVE,'r:gz') as tar:
        members=tar.getmembers()
        for m in members:
            pp=Path(m.name)
            assert pp.parts[0]=='M4_v2' and '..' not in pp.parts and (m.isfile() or m.isdir())
        tar.extractall(stage,filter='data')
    ROOT=NEW
assert (ROOT/'FINAL_TEST_FREEZE.json').is_file()
OUT=Path('/workspace/M4_FIX2_GQDP'); OUT.mkdir(parents=True,exist_ok=True)
print('SOURCE PROJECT:',ROOT)
print('FIX2 OUTPUT DIRECTORY:',OUT)

## 3. Restore standalone, inspected diagnostic code

The two runner programs below are embedded in this notebook. They reproduce the manuscript's documented GQDP algebra from the already provided sources; they do **not** import code from `anon_user`.

In [ ]:
from pathlib import Path
import base64,hashlib
code_bytes=base64.b64decode('...')
target=OUT/'m4_fix2_gqdp.py'
target.write_bytes(code_bytes)
assert hashlib.sha256(target.read_bytes()).hexdigest()=='22e9bc240c96904c30d0d3c5a66b4851254194a9e2ab51cc0bc46da777264161'
print('WROTE + SHA-256 VERIFIED:',target)

In [ ]:
from pathlib import Path
import base64,hashlib
code_bytes=base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZXN1bWFibGUgdHdvLUdQVSBzY2hlZHVsZXIgZm9yIHNlcGFyYXRlIHNlZWQvcm9sZSBkaWFnbm9zdGljIHByb2Jlcy4iIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBhcmdwYXJzZSwgY29sbGVjdGlvbnMsIGpzb24sIG9zLCBzdWJwcm9jZXNzLCBzeXMsIHRpbWUsIGRhdGV0aW1lLCB0cmFjZWJhY2sKClRBU0tTID0gWyhzLHIpIGZvciBzIGluIFsyMDI2MTEwMSwyMDI2MTEwMiwyMDI2MTEwM10gZm9yIHIgaW4gWydPJywnVSddXQoKCmRlZiBsb2cobGluZSk6CiAgICBtc2c9Zid7ZGF0ZXRpbWUuZGF0ZXRpbWUubm93KGRhdGV0aW1lLnRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCl9IHtsaW5lfScKICAgIHByaW50KG1zZyxmbHVzaD1UcnVlKQoKCmRlZiBtYWluKCk6CiAgICBhcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tc2NyaXB0JyxyZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLXJvb3QnLHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoJy0tb3V0JyxyZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCctLWdwdXMnLHR5cGU9aW50LGRlZmF1bHQ9MikKICAgIGFwLmFkZF9hcmd1bWVudCgnLS1iYXRjaCcsdHlwZT1pbnQsZGVmYXVsdD04KQogICAgYXJncz1hcC5wYXJzZV9hcmdzKCkKICAgIGZyb20gaW1wb3J0bGliLnV0aWwgaW1wb3J0IHNwZWNfZnJvbV9maWxlX2xvY2F0aW9uLG1vZHVsZV9mcm9tX3NwZWMKICAgICMgUmVhZC1vbmx5IGltcG9ydCB0byBjb25maXJtIHNvdXJjZS9kYXRhc2V0IGZyZWV6ZSBiZWZvcmUgcGFpZCBHUFUgd29yay4KICAgIHNwZWM9c3BlY19mcm9tX2ZpbGVfbG9jYXRpb24oJ200ZGlhZycsYXJncy5zY3JpcHQpO21vZD1tb2R1bGVfZnJvbV9zcGVjKHNwZWMpO3NwZWMubG9hZGVyLmV4ZWNfbW9kdWxlKG1vZCkKICAgIG1vZC5wcmVmbGlnaHQoUGF0aChhcmdzLnJvb3QpKQogICAgb3V0PVBhdGgoYXJncy5vdXQpO291dC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIChvdXQvJ2xvZ3MnKS5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgam9icz1jb2xsZWN0aW9ucy5kZXF1ZShUQVNLUykKICAgIGFjdGl2ZT17fQogICAgZG9uZT1zZXQoKQogICAgZm9yIHNlZWQscm9sZSBpbiBUQVNLUzoKICAgICAgICB0YXJnZXQ9b3V0L2Yne3NlZWR9X3tyb2xlfV9ncWRwMTYuanNvbicKICAgICAgICBpZiB0YXJnZXQuZXhpc3RzKCk6CiAgICAgICAgICAgIG9iaj1qc29uLmxvYWRzKHRhcmdldC5yZWFkX3RleHQoKSkKICAgICAgICAgICAgaWYgb2JqWydzb3VyY2Vfc2hhMjU2J10hPW1vZC5TT1VSQ0VfU0hBIG9yIGxlbihvYmpbJ3Jvd3MnXSkhPTMyOnJhaXNlIFJ1bnRpbWVFcnJvcihmJ1VudmVyaWZpZWQgcHJpb3IgcmVzdWx0IHt0YXJnZXR9JykKICAgICAgICAgICAgZG9uZS5hZGQoKHNlZWQscm9sZSkpCiAgICBqb2JzPWNvbGxlY3Rpb25zLmRlcXVlKFt6IGZvciB6IGluIGpvYnMgaWYgeiBub3QgaW4gZG9uZV0pCiAgICBsb2coZidTVEFSVCBGSVgyIDE2IG1hdHJpY2VzIC8gdHdvIHF1YW50aXplcnMgLyAzIHNlZWRzIC8gTyxVOyBjb21wbGV0ZWQge2xlbihkb25lKX0vNicpCiAgICBuPW1pbihhcmdzLmdwdXMsaW50KG9zLmVudmlyb24uZ2V0KCdNNF9HUFVfQ09VTlQnLGFyZ3MuZ3B1cykpKQogICAgYXNzZXJ0IG4+PTEKICAgIHRyeToKICAgICAgd2hpbGUgam9icyBvciBhY3RpdmU6CiAgICAgICAgZm9yIGdwdSBpbiByYW5nZShuKToKICAgICAgICAgICAgaWYgZ3B1IG5vdCBpbiBhY3RpdmUgYW5kIGpvYnM6CiAgICAgICAgICAgICAgICBzZWVkLHJvbGU9am9icy5wb3BsZWZ0KCkKICAgICAgICAgICAgICAgIGVudj1vcy5lbnZpcm9uLmNvcHkoKTtlbnZbJ0NVREFfVklTSUJMRV9ERVZJQ0VTJ109c3RyKGdwdSkKICAgICAgICAgICAgICAgIGxvZ19maWxlPW91dC8nbG9ncycvZid7c2VlZH1fe3JvbGV9X2dwdXtncHV9LmxvZycKICAgICAgICAgICAgICAgIGY9b3Blbihsb2dfZmlsZSwndycsZW5jb2Rpbmc9J3V0Zi04JykKICAgICAgICAgICAgICAgIGNtZD1bc3lzLmV4ZWN1dGFibGUsYXJncy5zY3JpcHQsJ3dvcmtlcicsJy0tcm9vdCcsYXJncy5yb290LCctLW91dCcsYXJncy5vdXQsJy0tc2VlZCcsc3RyKHNlZWQpLCctLXJvbGUnLHJvbGUsJy0tYmF0Y2gnLHN0cihhcmdzLmJhdGNoKV0KICAgICAgICAgICAgICAgIHByb2M9c3VicHJvY2Vzcy5Qb3BlbihjbWQsc3Rkb3V0PWYsc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULGVudj1lbnYsc3RhcnRfbmV3X3Nlc3Npb249VHJ1ZSkKICAgICAgICAgICAgICAgIGFjdGl2ZVtncHVdPShwcm9jLGYsc2VlZCxyb2xlLHRpbWUudGltZSgpLGxvZ19maWxlKQogICAgICAgICAgICAgICAgbG9nKGYnU1RBUlQge3NlZWR9L3tyb2xlfSBHUFUge2dwdX0gUElEPXtwcm9jLnBpZH0nKQogICAgICAgIGlmIGFjdGl2ZToKICAgICAgICAgICAgdGltZS5zbGVlcCgzKQogICAgICAgIGZvciBncHUsKHByb2MsZixzZWVkLHJvbGUsc3RhcnRlZCxwYXRoKSBpbiBsaXN0KGFjdGl2ZS5pdGVtcygpKToKICAgICAgICAgICAgcmM9cHJvYy5wb2xsKCkKICAgICAgICAgICAgaWYgcmMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBmLmZsdXNoKCk7Zi5jbG9zZSgpCiAgICAgICAgICAgICAgICBpZiByYyE9MDoKICAgICAgICAgICAgICAgICAgICBsb2coZidFUlJPUiB7c2VlZH0ve3JvbGV9IEdQVSB7Z3B1fSByYz17cmN9OyBzZWUge3BhdGh9JykKICAgICAgICAgICAgICAgICAgICBmb3IgZywocCxmZiwqXykgaW4gYWN0aXZlLml0ZW1zKCk6CiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGchPWdwdSBhbmQgcC5wb2xsKCkgaXMgTm9uZTpwLnRlcm1pbmF0ZSgpCiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdXb3JrZXIgZmFpbGVkLiBJbnNwZWN0IGxhc3QgMTAwIGxpbmVzIG9mIGl0cyBsb2cuIE91dHB1dHMgZnJvbSBzdWNjZXNzZnVsIHRhc2tzIHByZXNlcnZlZC4nKQogICAgICAgICAgICAgICAgcmVzdWx0PW91dC9mJ3tzZWVkfV97cm9sZX1fZ3FkcDE2Lmpzb24nCiAgICAgICAgICAgICAgICBpZiBub3QgcmVzdWx0LmV4aXN0cygpIG9yIGxlbihqc29uLmxvYWRzKHJlc3VsdC5yZWFkX3RleHQoKSlbJ3Jvd3MnXSkhPTMyOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmJ1dvcmtlciByZXR1cm5lZCBzdWNjZXNzIHdpdGhvdXQgY29tcGxldGUsIHZhbGlkYXRlZCBmaWxlOiB7cmVzdWx0fScpCiAgICAgICAgICAgICAgICBkb25lLmFkZCgoc2VlZCxyb2xlKSkKICAgICAgICAgICAgICAgIGxvZyhmJ0RPTkUge3NlZWR9L3tyb2xlfSBHUFUge2dwdX0gbWludXRlcz17KHRpbWUudGltZSgpLXN0YXJ0ZWQpLzYwOi4yZn0gKHtsZW4oZG9uZSl9LzYpJykKICAgICAgICAgICAgICAgIGRlbCBhY3RpdmVbZ3B1XQogICAgICBsb2coJ0FMTCBTSVggVEFTS1MgQ09NUExFVEU7IEFHR1JFR0FURSBQRU5ESU5HJykKICAgIGZpbmFsbHk6CiAgICAgIGZvciBncHUsam9iIGluIGxpc3QoYWN0aXZlLml0ZW1zKCkpOgogICAgICAgICAgcCxmLCpfPWpvYgogICAgICAgICAgaWYgcC5wb2xsKCkgaXMgTm9uZTpwLnRlcm1pbmF0ZSgpCiAgICAgICAgICB0cnk6Zi5jbG9zZSgpCiAgICAgICAgICBleGNlcHQ6cGFzcwoKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiAgICB0cnk6bWFpbigpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHRyYWNlYmFjay5wcmludF9leGMoKTtzeXMuZXhpdCgxKQo=')
target=OUT/'m4_fix2_scheduler.py'
target.write_bytes(code_bytes)
assert hashlib.sha256(target.read_bytes()).hexdigest()=='64d6dcaf01f52355a0ea3a8ed966c4f88d5562498e95eefd2b1ff20b1c6f6280'
print('WROTE + SHA-256 VERIFIED:',target)

## 4. CPU/data and GPU preflight — must pass before billing-intensive execution

In [ ]:
import subprocess,sys
r=subprocess.run([sys.executable,str(OUT/'m4_fix2_gqdp.py'),'preflight','--root',str(ROOT)],capture_output=True,text=True)
print(r.stdout)
if r.returncode: print(r.stderr)
assert r.returncode==0,'Aborted: source/adapter/freeze preflight failed'

In [ ]:
import torch,bitsandbytes as bnb,importlib.util
print('GPU count:',torch.cuda.device_count())
assert torch.cuda.device_count()>=1,'Requires NVIDIA GPU'
for i in range(torch.cuda.device_count()):
    print(i,torch.cuda.get_device_name(i),torch.cuda.get_device_capability(i))
    assert torch.cuda.get_device_capability(i)[0]>=8,'Use native BF16 A100/H100 etc.'
spec=importlib.util.spec_from_file_location('m4fix2',str(OUT/'m4_fix2_gqdp.py'))
m4=importlib.util.module_from_spec(spec);spec.loader.exec_module(m4)
W=torch.nn.Linear(64,64,bias=False).to(device='cuda',dtype=torch.bfloat16)
for kind in ('nf4','int8'):
    packed,eps=m4.packed_isolated(W,kind)
    with torch.inference_mode():
        got=packed(torch.randn(2,64,device='cuda',dtype=torch.bfloat16))
    assert got.shape==(2,64) and torch.isfinite(got).all()
    assert eps.shape==W.weight.shape and eps.abs().sum().item()>0
    print('PASS: genuine isolated packed',kind,'module:',type(packed).__name__)
    del packed,eps
print('PASS: READY FOR M4 FIX2 GQDP')

## 5. Launch six **evaluation-only** jobs across available GPUs

This starts a detached scheduler so closing the JupyterLab browser does not terminate the process. It stops on any worker error and preserves validated successful jobs. **Run the launch cell only once.** If a run has already started, use the monitoring cell instead.

In [ ]:
import os,sys,subprocess
from pathlib import Path
n_gpus=min(2,torch.cuda.device_count())
master=OUT/'M4_FIX2_MASTER.log'
pidfile=OUT/'M4_FIX2_MASTER.pid'
if pidfile.exists():
    old_pid=int(pidfile.read_text().strip())
    try: os.kill(old_pid,0); raise RuntimeError(f'Scheduler already running: PID={old_pid}; do NOT relaunch')
    except ProcessLookupError: pass
cmd=[sys.executable,str(OUT/'m4_fix2_scheduler.py'),'--script',str(OUT/'m4_fix2_gqdp.py'),
    '--root',str(ROOT),'--out',str(OUT),'--gpus',str(n_gpus),'--batch','8']
with open(master,'a',encoding='utf-8') as f:
    p=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT,start_new_session=True,cwd=str(OUT))
pidfile.write_text(str(p.pid))
print('SCHEDULER STARTED',p.pid,'on',n_gpus,'GPU(s)')
print('Logs:',master)

## 6. Monitor as needed (rerun safely)

In [ ]:
from pathlib import Path
import subprocess
print('\n'.join((OUT/'M4_FIX2_MASTER.log').read_text().splitlines()[-30:]))
completed=list(OUT.glob('2026110*_?_gqdp16.json'))
print(f'COMPLETED TASKS: {len(completed)}/6 (32 matrix/quantizer records each)')
for logfile in sorted((OUT/'logs').glob('*.log')):
    last=logfile.read_text(errors='replace').splitlines()[-1:]
    print(logfile.name, last[0][-180:] if last else '')

## 7. Recompute correlations and export the standalone results

Only run once the monitor confirms 6/6 tasks. Computes 12 within-condition GQDP Spearman coefficients against dense-dequant GT plus 12 packed-GT coefficients (16 matrices each), clustered-by-depth exploratory bootstrap intervals, three-seed summaries and NF4–INT8 contrasts. The aggregate cannot run with missing jobs and never fabricates results.

In [ ]:
import subprocess,sys
proc=subprocess.run([sys.executable,str(OUT/'m4_fix2_gqdp.py'),'aggregate','--root',str(ROOT),'--out',str(OUT)],capture_output=True,text=True)
print(proc.stdout)
if proc.returncode:print(proc.stderr)
assert proc.returncode==0,'Not all six task results ready or invalid'
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUT/'M4_FIX2_group_correlations.csv'))

In [ ]:
import zipfile,hashlib,json
from pathlib import Path
outfile=Path('/workspace/M4_FIX2_GQDP_RESULTS.zip')
required=['M4_FIX2_all192.csv','M4_FIX2_group_correlations.csv',
          'M4_FIX2_quantizer_contrasts.csv','M4_FIX2_summary.json',
          'm4_fix2_gqdp.py','m4_fix2_scheduler.py','M4_FIX2_MASTER.log']
required += [f'{s}_{r}_gqdp16.json' for s in (20261101,20261102,20261103) for r in ('O','U')]
with zipfile.ZipFile(outfile,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for name in required:
        p=OUT/name
        assert p.is_file(),f'Missing {p}'
        z.write(p,arcname=name)
    for p in (OUT/'logs').glob('*.log'):z.write(p,arcname='logs/'+p.name)
with zipfile.ZipFile(outfile) as z: assert z.testzip() is None
print('DOWNLOAD THIS RESULTS FILE:',outfile)
print('SIZE MB:',round(outfile.stat().st_size/1e6,2))
print('SHA256:',hashlib.sha256(outfile.read_bytes()).hexdigest())
print('Upload to ChatGPT for independent scientific audit before writing the new M4 paragraphs.')